In [49]:
import pandas as pd

# Dataset Kaggle
df = pd.read_csv("../datasets/customer_support_tickets.csv")

df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [50]:
df.columns

Index(['Ticket ID', 'Customer Name', 'Customer Email', 'Customer Age',
       'Customer Gender', 'Product Purchased', 'Date of Purchase',
       'Ticket Type', 'Ticket Subject', 'Ticket Description', 'Ticket Status',
       'Resolution', 'Ticket Priority', 'Ticket Channel',
       'First Response Time', 'Time to Resolution',
       'Customer Satisfaction Rating'],
      dtype='str')

In [51]:
df["Ticket Subject"].value_counts().head(30)

Ticket Subject
Refund request              576
Software bug                574
Product compatibility       567
Delivery problem            561
Hardware issue              547
Battery life                542
Network problem             539
Installation support        530
Product setup               529
Payment issue               526
Product recommendation      517
Account access              509
Peripheral compatibility    496
Data loss                   491
Cancellation request        487
Display issue               478
Name: count, dtype: int64

In [52]:
# Création mapping catégories

category_mapping = {
    "Hardware issue": "hardware",
    "Display issue": "hardware",
    "Battery life": "hardware",

    "Software bug": "software",
    "Account access": "software",

    "Network problem": "network",
    "Peripheral compatibility": "network",

    "Product setup": "configuration",
    "Installation support": "configuration",
    "Product compatibility": "configuration",

    "Delivery problem": "delivery",

    "Refund request": "warranty",
    "Cancellation request": "warranty",

    "Payment issue": "billing",

    "Data loss": "performance",

    "Product recommendation": "inquiry"
}

df["problem_category"] = df["Ticket Subject"].map(category_mapping)

df[["Ticket Subject", "problem_category"]].head()

,Ticket Subject,problem_category
0,Product setup,configuration
1,Peripheral compatibility,network
2,Network problem,network
3,Account access,software
4,Data loss,performance


In [28]:
# Création mapping catégories

category_mapping = {
    "Hardware issue": "hardware",
    "Display issue": "hardware",
    "Battery life": "hardware",

    "Software bug": "software",
    "Account access": "software",

    "Network problem": "network",
    "Peripheral compatibility": "network",

    "Product setup": "configuration",
    "Installation support": "configuration",
    "Product compatibility": "configuration",

    "Delivery problem": "delivery",

    "Refund request": "warranty",
    "Cancellation request": "warranty",

    "Payment issue": "billing",

    "Data loss": "performance",

    "Product recommendation": "inquiry"
}

# Nouvelle colonne
df["problem_category"] = df["Ticket Subject"].map(category_mapping)

# Vérification
df[["Ticket Subject", "problem_category"]].head()

,Ticket Subject,problem_category
0,Product setup,configuration
1,Peripheral compatibility,network
2,Network problem,network
3,Account access,software
4,Data loss,performance


In [53]:
df["problem_category"].value_counts()

problem_category
configuration    1626
hardware         1567
software         1083
warranty         1063
network          1035
delivery          561
billing           526
inquiry           517
performance       491
Name: count, dtype: int64

In [54]:
# Combiner sujet + description

df["text"] = (
    df["Ticket Subject"].fillna('') + " " +
    df["Ticket Description"].fillna('')
)

ml_df = df[["text", "problem_category"]]

ml_df.head()

,text,problem_category
0,Product setup I'm having an issue with the {pr...,configuration
1,Peripheral compatibility I'm having an issue w...,network
2,Network problem I'm facing a problem with my {...,network
3,Account access I'm having an issue with the {p...,software
4,Data loss I'm having an issue with the {produc...,performance


In [55]:
ml_df = ml_df.dropna()

print(ml_df.shape)

(8469, 2)


In [57]:
custom_df = pd.read_csv("../datasets/custom_hp_dataset.csv")

custom_df.head()

,text,category
0,My HP printer cannot connect to the WiFi netwo...,network
1,The laptop keeps disconnecting from the intern...,network
2,I cannot connect my HP printer to the wireless...,network
3,The WiFi connection on my computer is unstable...,network
4,The Ethernet connection is not detected on my ...,network


In [58]:
custom_df = custom_df.rename(columns={
    "category": "problem_category"
})

In [59]:
custom_ml_df = custom_df[["text", "problem_category"]]

In [60]:
final_df = pd.concat(
    [ml_df, custom_ml_df],
    ignore_index=True
)

final_df = final_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print(final_df.shape)

(8649, 2)


In [61]:
X = final_df["text"]
y = final_df["problem_category"]

In [62]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [35]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(X_train_tfidf.shape)

(6775, 5000)


In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=7000,
    ngram_range=(1,2)
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [37]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.9976387249114522

Classification Report:

               precision    recall  f1-score   support

      billing       1.00      1.00      1.00       105
configuration       0.99      1.00      1.00       337
     delivery       1.00      1.00      1.00       113
     hardware       1.00      1.00      1.00       334
      inquiry       1.00      1.00      1.00       101
      network       1.00      0.99      1.00       190
  performance       0.98      1.00      0.99       102
     software       1.00      1.00      1.00       216
     warranty       1.00      0.99      1.00       196

     accuracy                           1.00      1694
    macro avg       1.00      1.00      1.00      1694
 weighted avg       1.00      1.00      1.00      1694



In [64]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000)

model.fit(X_train_tfidf, y_train)

print("Improved bilingual model trained!")

Improved bilingual model trained!


In [65]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print(classification_report(y_test, y_pred))

Accuracy: 0.9861271676300578
               precision    recall  f1-score   support

      billing       0.99      0.97      0.98       105
configuration       0.99      0.99      0.99       334
     delivery       1.00      0.94      0.97       107
     hardware       0.98      0.99      0.98       324
      inquiry       1.00      0.98      0.99       112
      network       0.97      1.00      0.98       202
  performance       1.00      0.99      0.99        96
     software       0.99      0.99      0.99       222
     warranty       0.99      0.99      0.99       228

     accuracy                           0.99      1730
    macro avg       0.99      0.98      0.99      1730
 weighted avg       0.99      0.99      0.99      1730



In [69]:
samples = [
    "the wifi not working",
    "le wifi ne fonctionne plus",
    "mon pc n'est plus sous garantie",
    "blue screen after update",
    "écran noir au démarrage",
    "le wifi ne fonctionne pas"
]

samples_tfidf = vectorizer.transform(samples)

predictions = model.predict(samples_tfidf)
for text, pred in zip(samples, predictions):
    print(text, "→", pred)

the wifi not working → network
le wifi ne fonctionne plus → hardware
mon pc n'est plus sous garantie → warranty
blue screen after update → hardware
écran noir au démarrage → hardware
le wifi ne fonctionne pas → network


In [70]:
import joblib

# Sauvegarde modèle
joblib.dump(
    model,
    "../models/complaint_classifier.pkl"
)

# Sauvegarde vectorizer
joblib.dump(
    vectorizer,
    "../models/tfidf_vectorizer.pkl"
)

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
